# pitcheezy Colab 러너

유일한 실행 진입점. **로직 없음** — 코드는 레포, Colab은 실행기.

순서: 파라미터 → clone/pull → `pip install -e .` → Drive 마운트 → W&B 키(Colab Secrets) → `run_experiment.py`

In [ ]:
#@title 파라미터
REPO_URL = "https://github.com/SongRoute/pitcheezy.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
CONFIG = "configs/p0/EXP-P0-A-001.yaml"  #@param {type:"string"}
SEED = 0  #@param {type:"integer"}

In [ ]:
#@title 레포 clone / pull
import os

if not os.path.isdir("/content/pitcheezy"):
    !git clone --quiet "$REPO_URL" /content/pitcheezy
%cd /content/pitcheezy
!git fetch --quiet
!git checkout --quiet "$BRANCH"
!git pull --ff-only

In [ ]:
#@title 패키지 설치
!pip install -q -e .

In [ ]:
#@title Drive 마운트 + 데이터·실행 경로
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/pitcheezy"
os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/runs", exist_ok=True)
os.environ["PITCHEEZY_DATA_DIR"] = f"{DRIVE_ROOT}/data"  # 레포 data/에는 versions.md만 커밋
os.environ["PITCHEEZY_RUNS_DIR"] = f"{DRIVE_ROOT}/runs"  # {실험ID}/{seed}/

In [ ]:
#@title W&B 로그인 (Colab Secrets)
# 좌측 🔑 Secrets에 WANDB_API_KEY를 등록하고 이 노트북의 접근을 허용한다. 키를 노트북·레포에 쓰지 않는다.
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

In [ ]:
#@title 실험 실행
!python scripts/run_experiment.py --config "$CONFIG" --seed "$SEED"

## 세션 만료 대비

- Colab Pro는 백그라운드 실행이 없어 탭이 닫히면 끊길 수 있다. 5시드 채택 판정은 시드 단위로 나눠 실행한다 (`SEED`만 바꿔 재실행)
- 러너는 `$PITCHEEZY_RUNS_DIR/{ID}/{seed}/`의 체크포인트에서 재개한다
- 해석·채택 판단은 노션 04 실험 로그에서만. 이 노트북 출력을 결과의 근거로 인용하지 않는다